# 00 · SHAP attribution across FTTL versions — real data

**What this notebook does NOT do: it never opens a model pickle.** A pickle only unpickles inside
the env it was serialised in, and the three FTTL versions have mutually incompatible library stacks
(`src/docs/ENV_MANAGEMENT.md`). So the split is:

| step | where it runs | what it produces |
|---|---|---|
| compute per-row φ | `src/envs/v<k>/.venv/bin/python src/scoring/attribute.py` — **once per version, in that version's own env** | `src/data/real/detection/<v>_attributions.parquet` + `<v>_attributions_meta.json` |
| **this notebook** | the shared analysis `.venv` (kernel `sfp-detection`) | tables + figures |

Run the compute step first — one command does all three versions:

```bash
python src/scoring/attribute_all.py --rows 5000 --background 500      # add --dry-run first
```

φ values are just numbers once they are on disk, so everything below is version-agnostic and the
dependency problem disappears. Nothing here hard-codes a path, a repo name or a column name —
they all come from `src/config.py` via `loaders.load()`.

**Read every number below at L0.** Feature names are the model's own raw column names. v1's 55
`make_*` columns and v2's 41 are *not* collapsed into one `make` — doing so would change the
concentration statistic this chapter rests on. Cross-version correspondence comes **only** from the
hand-confirmed mapping (`features/check_overlap.py` → `features/feature_overlap.json`), never from
name equality.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config           # noqa: E402  — real paths/columns live here and nowhere else
import schema           # noqa: E402
import figstyle         # noqa: E402
from loaders import load                       # noqa: E402
from estimator import concentration as conc    # noqa: E402

figstyle.apply()

SOURCE   = "real"
VERSIONS = list(config.VERSION_LABELS)   # trim to e.g. ["v2", "v3"] to compare a single pair
TOPN     = 15                            # features shown per version in the importance panels
TOPK     = 5                             # k for top-k share

print(f"repo root : {ROOT}")
print(f"versions  : {VERSIONS}  (source={SOURCE})")

## 1 · What is actually on disk, and was it produced comparably?

A version is skipped here — loudly — rather than silently dropped, so a missing arm can never be
mistaken for a version with no signal.

`require_comparable()` then **raises** if the versions were attributed under different SHAP
backends. Interventional TreeSHAP (against a fixed shared background) and tree-path-dependent
TreeSHAP (against each tree's own cover statistics) answer different questions; mixed, they produce
a perfectly plausible-looking difference that is an artefact of the reference distribution.

In [ ]:
avail, missing = {}, {}
for v in VERSIONS:
    try:
        d = load(v, SOURCE)
        d.attributions                     # forces the read; raises with the fix if absent
        avail[v] = d
    except FileNotFoundError as exc:
        missing[v] = str(exc).replace("\n", " ")

for v, why in missing.items():
    print(f"[{v}] NOT attributed yet — {why}\n")

assert avail, ("no version has attributions on disk. Run:\n"
               "    python src/scoring/attribute_all.py --dry-run\n"
               "then without --dry-run.")

metas = {v: d.attribution_meta for v, d in avail.items()}
conc.require_comparable(metas)             # raises on mixed backends; warns on mixed row sets

FIELDS = ("backend", "perturbation", "model_output", "estimator", "feature_order",
          "n_rows", "n_features", "background_n", "base_value")
pd.DataFrame({v: {k: m.get(k) for k in FIELDS} for v, m in metas.items()}).T

## 2 · Where each version puts its attribution mass

One panel per version, top-`TOPN` by mean |φ| — **not** a grouped bar chart, because the versions do
not share a feature set (v2's headline feature `location_Home` does not exist in v1 or v3 at all).
Ranking them on one axis would imply a correspondence that is not there.

The panel title carries `top N of <total>`: a version with far more columns can look "flatter" for
that reason alone, which is exactly what §4's evenness ratios control for.

In [ ]:
MABS = {v: d.mean_abs_shap for v, d in avail.items()}     # mean|φ| per feature, descending

n = len(MABS)
fig, axes = plt.subplots(1, n, figsize=(max(figstyle.FIG_1[0], 4.0 * n), 0.30 * TOPN + 1.6))
for ax, (v, m), colour in zip(np.atleast_1d(axes), MABS.items(), figstyle.SERIES):
    top = m.head(TOPN)[::-1]
    ax.barh(np.arange(len(top)), top.values, color=colour)
    ax.set_yticks(np.arange(len(top)))
    ax.set_yticklabels(top.index, fontsize=7)
    ax.set_title(f"{v} — top {TOPN} of {len(m)}")
    ax.set_xlabel("mean |φ|  (log-odds)")

fig.suptitle("Attribution mass by feature — L0 raw names, per version")
fig.tight_layout()
figstyle.save(fig, "00_real_shap_global_importance")
plt.show()

pd.DataFrame({v: m.head(TOPN).round(4).reset_index().agg(
    lambda r: f"{r['index']}  {r[0]:.4f}", axis=1) for v, m in MABS.items()})

## 3 · How much of this is even comparable — the hand-confirmed mapping

The versions encode the same concepts under **different L0 names** (v2 `veh_total_loss` ≡ v3 `Fttl`;
`ReportedDate` ≡ `ReportedDate_CLAIM`), so a raw string intersection **undercounts** what is truly
shared — it is shown below only as an exhibit of that encoding divergence. The comparison basis is
`features/feature_overlap.json`: the cross-version correspondence confirmed **by hand** in
`features/common_features_260804.xlsx` and exported by `features/check_overlap.py` (~70 entries;
a subset spans all three versions, others pair v2–v3 only). No fuzzy matching anywhere.

If the JSON has not been built yet (it requires the Excel, on the company laptop), the notebook
falls back to the raw intersection and says so — every downstream "restricted" number is then an
undercount and must not be reported.

In [ ]:
import json

names = {v: set(m.index) for v, m in MABS.items()}

# ── (a) raw string-equality intersection — the encoding-divergence EXHIBIT, not the basis
RAW_INTER = set.intersection(*names.values()) if len(names) > 1 else set(next(iter(names.values())))
print(f"raw L0 string intersection across {list(names)}: {len(RAW_INTER)} features")
print("  (expected to be small — same concepts, different encodings. Comparison uses the")
print("   hand-confirmed mapping below, never name equality.)\n")

# ── (b) the hand-confirmed mapping: {"<idx>": {"v1": name, "v2": name, "v3": name}, ...}
overlap_json = ROOT / "features" / "feature_overlap.json"
MAPPING = {}     # idx -> {version: that version's own L0 name}
SHARED = {}      # version -> the set of ITS OWN names for rows mapped in every loaded version
if overlap_json.exists():
    MAPPING = {int(k): row for k, row in json.loads(overlap_json.read_text()).items()}
    have = list(MABS)
    # a row is usable only if every loaded version has a name there AND that name actually
    # appears in that version's attribution columns (typo/regeneration guard)
    full_rows = {k: row for k, row in MAPPING.items()
                 if all(v in row and row[v] in names[v] for v in have)}
    dropped = [k for k, row in MAPPING.items()
               if all(v in row for v in have) and k not in full_rows]
    if dropped:
        print(f"⚠ {len(dropped)} mapped rows name features ABSENT from the attributions "
              f"{dropped[:8]}{'…' if len(dropped) > 8 else ''} — stale mapping or wrong matrix; "
              f"resolve before reporting.")
    SHARED = {v: {row[v] for row in full_rows.values()} for v in have}
    print(f"hand-confirmed mapping: {len(MAPPING)} entries; {len(full_rows)} usable across "
          f"{have} and present in the attributions")
    for v in have:
        held = float(MABS[v][MABS[v].index.isin(SHARED[v])].sum() / MABS[v].sum()) if SHARED[v] else 0.0
        print(f"  {v}: {len(names[v]):>4} L0 features · {len(SHARED[v]):>3} mapped · "
              f"mapped set holds {held:.1%} of its attribution mass")
    if len(have) > 1:
        pairs = pd.DataFrame(
            [[sum(1 for row in MAPPING.values() if a in row and b in row) for b in have]
             for a in have], index=have, columns=have)
        print("\nmapped features per version pair (diagonal = that version's mapped total):")
        display(pairs)
else:
    print(f"{overlap_json} not built yet — run features/check_overlap.py (needs the Excel; "
          f"company laptop).\nFalling back to the RAW intersection: every restricted number "
          f"below is an UNDERCOUNT — do not report it.")
    SHARED = {v: set(RAW_INTER) for v in MABS}

# the restricted comparisons are sized by construction: every SHARED[v] has the same length
N_SHARED = min((len(s) for s in SHARED.values()), default=0)

## 4 · Concentration — the statistic the SFP claim rests on

Two tables, and they answer different questions.

**All features.** Each version's own full profile. `richness_D0` differs because the versions *have*
different numbers of columns, so `exp_shannon_D1` and `inv_simpson_D2` are not directly comparable
in raw form — read `evenness_D1_D0` / `evenness_D2_D0`, which are unit-free.

**Restricted to the mapped features.** Each version keeps **its own L0 names** for the concepts the
hand-confirmed mapping links (§3); mass outside them is dropped and the shares renormalised, so the
feature count is identical across versions by construction and `D1`/`D2` *are* directly comparable.
This is the like-for-like number. It answers a narrower question — "among the features the versions
demonstrably share, has the mass concentrated?" — and that narrowing must be stated wherever it is
reported, together with the mapped-mass coverage from §3 (a small covered fraction weakens the
comparison even though it runs).

Direction of the hypothesis: if the loop is concentrating the model's reasoning onto the
fast-track drivers, later versions should show **lower** D1/D2 and **higher** Simpson / Gini /
top-k share.

In [ ]:
full = conc.profile_table(MABS, k=TOPK)
print("All features (D0 differs by construction — compare the evenness ratios, not D1/D2)")
display(full.round(4))

In [ ]:
if N_SHARED >= 5 and len(MABS) > 1:
    shared = {v: m[m.index.isin(SHARED[v])] for v, m in MABS.items()}
    restricted = conc.profile_table(shared, k=TOPK)
    print(f"Restricted to the {N_SHARED} mapped features — D1/D2 directly comparable.")
    print("(Each version restricted to ITS OWN L0 names for the mapped concepts; "
          "shares renormalised within the mapped set.)")
    display(restricted.round(4))
else:
    restricted = None
    print(f"mapped coverage is {N_SHARED} features — too thin for a restricted comparison. "
          f"Report the per-version profiles only, and the overlap itself as the finding.")

### 4b · The configuration that produced those numbers

`problem.md` §1.4c is blunt about this: **no adjacent version pair shares a configuration.** v2
regularises through penalties (`reg_alpha=20`, `gamma=15`, deep trees), v3 through tree structure
(`max_depth=3`, `max_leaves=18`, `min_child_weight=44`) — and **L1 concentrates feature importance
by construction**. So a rise in concentration has a competing mechanical explanation that predicts
the same direction as the loop.

The rule adopted there is that every version's configuration is printed beside its concentration
figures. `attribute.py` records the estimator's params in the meta so this table cannot drift from
the numbers above.

In [ ]:
KNOBS = ("n_estimators", "max_depth", "max_leaves", "min_child_weight", "learning_rate", "eta",
         "reg_alpha", "reg_lambda", "gamma", "subsample", "colsample_bytree",
         "scale_pos_weight", "eval_metric", "objective")

params = {v: m.get("estimator_params") or {} for v, m in metas.items()}
if any(params.values()):
    cfg = pd.DataFrame({v: {k: p.get(k, "—") for k in KNOBS} for v, p in params.items()})
    display(cfg)
    if len(params) > 1:
        differs = [k for k in KNOBS if len({str(p.get(k)) for p in params.values()}) > 1]
        if differs:
            print(f"knobs that DIFFER: {differs}\n"
                  "The concentration difference above is confounded with these — report it as "
                  "*consistent with* the loop, not as identifying it, unless a matched-"
                  "hyperparameter refit or a sensitivity sweep bounds them (problem.md §1.4c i–iii).")
        else:
            print("these versions are configuration-matched on the knobs above — the "
                  "regularisation confound of problem.md §1.4c does not apply to this pair.")
else:
    print("no estimator params in the meta — re-run attribute_all.py to capture them "
          "(they can only be read where the pickle opens).")

In [ ]:
# Diversity profile: Dq over q. q=0 counts features, q→1 weights by share, q≥2 by dominance.
# Curves that CROSS mean the versions differ in the tail and in the drivers in opposite directions
# — a single q would hide that.
basis = restricted is not None
curves = {v: conc.hill_curve(m[m.index.isin(SHARED[v])] if basis else m) for v, m in MABS.items()}

fig, ax = plt.subplots(figsize=figstyle.FIG_1)
for (v, curve), colour in zip(curves.items(), figstyle.SERIES):
    ax.plot(curve.index, curve.values, color=colour, label=v)
ax.set_xlabel("order  q")
ax.set_ylabel("effective number of features  Dq")
ax.set_yscale("log")
ax.set_title("Attribution diversity profile" + (f" — {N_SHARED} mapped features" if basis
                                               else " — each version's own features"))
ax.legend(title="version")
fig.tight_layout()
figstyle.save(fig, "00_real_shap_hill_profile")
plt.show()

## 5 · Concentration either side of the fast-track cutoff

The SFP mechanism only operates on claims that were **scrapped** — those are the rows whose label
was forced. So the sharper version of the question is whether concentration differs between the
scrapped and the garage-assessed side of that version's own rule (v1 segmented on mobility, v2
piecewise in time, v3 global — `threshold.apply` dispatches; nothing here assumes a single τ).

This is descriptive, not causal: the two sides differ in case-mix as well as in treatment. It is
the input to the estimator layer, not a result on its own.

In [ ]:
try:
    rows = {}
    for v, d in avail.items():
        frame = d.frame[[schema.CLAIM_ID]].copy()
        frame["decision"] = d.decisions
        att = d.attributions.merge(frame, on=schema.CLAIM_ID, how="inner")
        if att.empty:
            print(f"[{v}] no attributed claim appears in the scored frame — skipping")
            continue
        for side, sub in att.groupby("decision"):
            label = "scrapped" if side == 1 else "garage"
            if len(sub) < 50:
                print(f"[{v}] {label}: only {len(sub)} rows — too thin, skipping")
                continue
            m = conc.mean_abs(sub.drop(columns=["decision"]), id_col=schema.CLAIM_ID)
            if N_SHARED >= 5:
                m = m[m.index.isin(SHARED[v])]    # this version's OWN names for the mapped set
            rows[(v, label)] = {**conc.profile(m, k=TOPK), "n": len(sub)}
    by_side = pd.DataFrame(rows).T
    display(by_side.round(4))
except FileNotFoundError as exc:
    by_side = None
    print(f"scores/log not available yet, so decisions cannot be reproduced — {exc}")

## Notes, and what would invalidate this

- **Backend.** If §1 reports `perturbation = tree_path_dependent`, the comparison is weaker: the
  reference is each tree's own cover statistics, so part of any cross-version difference is a
  difference in training distributions. Prefer `--backend shap` with a shared background; use
  `native` only when a frozen version env genuinely cannot take the `shap` dependency, and use the
  *same* backend for every version.
- **Rows.** The v2 and v3 windows are ~2¾ years apart, so a shared claim set is generally
  impossible — `attribute_all.py` falls back to `--per-version-sample`, and every difference here
  is then confounded with case-mix. That caveat is standing, not conditional.
- **Windows.** v2 and v3 were trained years apart (README), so concentration differences are
  *descriptive* of the fitted functions, not evidence of the loop on their own — the
  identification argument is the DiD in `04_02`, and its parallel-trends assumption is the thing
  to defend.
- **The mapping is the only bridge.** If §3 fell back to the raw string intersection, nothing
  restricted may be reported. And if a future edit ever ranks `make` rather than `make_FORD`, the
  number has changed meaning — collapse never happens here; correspondence comes only from
  `features/feature_overlap.json` (hand-confirmed; typo-checked against the registries by
  `features/check_overlap.py`).